# Models Pipeline

This notebook is the single restartable entry point. Reusable implementation lives in `src/`, while completed annual refits and diagnostics are cached by model ID and signature.

## Model roster

The retained models cover conventional neural and non-neural benchmarks; the LightGBM 20/40/60/80/100 characteristic-breadth comparison; exact-calendar lagged LightGBM models; matched MLP and DeepSets models using 40 characteristics; supporting Core20 DeepSets models; and four validation-weighted hybrids. `NN2_20` and `NN2_40` use hidden-layer widths `[32, 16]`, `NN3_20` uses `[32, 16, 8]`, and `NN4_20` uses `[32, 16, 8, 4]`. These are GKX-style benchmarks rather than exact replications because this project uses a different information set and sample. `MLP_40` remains the monthly-panel no-market-context control for `DEEPSET_40`. `DEEPSET_40_LAG1` adds current and exact one-month-lagged characteristics, while `DEEPSET_40_DYNAMIC` additionally includes rank velocities.

## Fixed design decisions

- The target is decimal next-month excess return, `ret_exc_lead1m`.
- The universe is USA stocks with valid PERMNO and size group micro/small/large/mega; nano stocks are excluded.
- Characteristics are ranked within the complete eligible monthly cross-section and mapped to `[-1, 1]` before target availability is inspected. Missing targets are masked from fitting and evaluation calculations but do not change the month-t ranking universe.
- Exact-calendar lags are joined by PERMNO. Missing prior months receive neutral values plus a zero availability flag.
- The rolling design uses 15 training years, 4 validation years, one temporally held-out test year, and annual refits from 1999 through 2024.
- Validation MSE selects hyperparameters and early stopping. Test outcomes never affect fitting or model selection.
- Primary evaluation is pooled GKX OOS R-squared and the equal-weighted D10-D1 portfolio. Rank IC, calibration, robust R-squared, monotonicity, alternative portfolios, transaction costs, universe sensitivity, and an adverse missing-return stress are supporting diagnostics.
- Constant-forecast months hold cash; PERMNO is only a deterministic tie-break when a genuine signal exists.
- Each model/refit has an independent signature and completion marker. Compatible results load without retraining or overwriting.

## 1. Runtime and project setup

The notebook file remains local and is edited in VS Code. When the selected kernel is Google Colab, this cell mounts Google Drive and uses the Drive copy of the project for source code, data, checkpoints, predictions, and diagnostics. With a normal local VS Code kernel, it uses the local Windows project folder instead. This storage choice does not change model definitions or experiment signatures.

In [ ]:
import os
import sys
import importlib
from pathlib import Path

LOCAL_PROJECT_DIR = Path(r"C:\Users\sandh\OneDrive\Documents\Coding\FDS Project")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/FDS Project")

try:
    from google.colab import drive
except ImportError:
    RUNNING_IN_COLAB = False
    PROJECT_DIR = LOCAL_PROJECT_DIR
    RUNTIME = "Local VS Code kernel"
else:
    RUNNING_IN_COLAB = True
    # Safe under Run all: mounts Drive when needed and reuses an existing mount.
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = DRIVE_PROJECT_DIR
    RUNTIME = "Google Colab kernel in VS Code"

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Project folder was not found: {PROJECT_DIR}\n"
        "If this is Colab, confirm that Drive is mounted and that the folder name matches exactly."
    )
if not (PROJECT_DIR / "src").is_dir():
    raise FileNotFoundError(f"The project src folder was not found under: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
# Always give this project priority over stale or similarly named packages.
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()

# Verify now, before any model configuration is evaluated.
import src
src_file = Path(src.__file__).resolve()
if PROJECT_DIR.resolve() not in src_file.parents:
    raise RuntimeError(f"Imported src from the wrong location: {src_file}")

print("Notebook file: local Models_Pipeline.ipynb")
print("Runtime:", RUNTIME)
print("Project files and outputs:", PROJECT_DIR)
print("Imported src from:", src_file)

## 2. Experiment configuration

Usually this is the only cell to edit. Use a new `experiment_id` after changing the universe, feature definition, target, preprocessing, or rolling schedule. The same experiment can safely be rerun or extended with new registered models.

In [ ]:
if 'PROJECT_DIR' not in globals():
    raise RuntimeError('Run the Runtime and project setup cell first, or use Run all.')

from src.config import ExperimentConfig
from src.models import MODEL_REGISTRY

DATA_PATH = PROJECT_DIR / 'jkp_USA_100chars_1980_2024.parquet'
OUTPUT_DIR = PROJECT_DIR / 'model_runs'
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Raw data file not found: {DATA_PATH}')

MODEL_ROSTER = (
    'LASSO_20', 'LGBM_20', 'XGBOOST_20', 'NN2_20', 'NN2_40', 'NN3_20', 'NN4_20',
    'LGBM_40', 'LGBM_60', 'LGBM_80', 'LGBM_100',
    'LGBM_20_LAG1', 'LGBM_20_LAG2',
    'LGBM_40_LAG1', 'LGBM_40_LAG2',
    'MLP_40', 'DEEPSET_40', 'DEEPSET_40_LAG1', 'DEEPSET_40_DYNAMIC',
    'DEEPSET_20', 'DEEPSET_20_LAG1', 'DEEPSET_20_DYNAMIC',
    'HYBRID_LGBM20_DEEPSET20', 'HYBRID_MLP40_DEEPSET40',
    'HYBRID_LGBM40_DEEPSET40', 'HYBRID_LGBM40_DEEPSET40_DYNAMIC',
)
# Hybrid components use years 1--3 of validation for early stopping;
# the convex combination is estimated on validation year 4.
SELECTED_MODELS = MODEL_ROSTER

CONFIG = ExperimentConfig(
    experiment_id='core20_benchmarks_v1',
    project_dir=PROJECT_DIR,
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    selected_models=SELECTED_MODELS,
    seed=42,
    use_gpu=True,
)
CONFIG.validate()

import torch
if CONFIG.use_gpu and not torch.cuda.is_available():
    raise RuntimeError('GPU requested but unavailable. In Colab choose Runtime > Change runtime type > T4 GPU, then reconnect.')
print('Torch device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Selected models:', list(CONFIG.selected_models))
print('Run directory:', CONFIG.run_dir)

## 3. Preflight checks

This verifies the Drive project files, model registry and deterministic feature-construction checks without loading the full panel. The runner loads and prepares the data once in the next section.

In [ ]:
import unittest
from src.self_checks import run_framework_self_checks

required_files = (
    DATA_PATH,
    PROJECT_DIR / 'src' / 'config.py',
    PROJECT_DIR / 'src' / 'models.py',
    PROJECT_DIR / 'src' / 'runner.py',
    PROJECT_DIR / 'src' / 'self_checks.py',
    PROJECT_DIR / 'tests' / 'test_pipeline.py',
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Required Drive project files are missing: {missing_files}')
unknown_models = sorted(set(CONFIG.selected_models) - set(MODEL_REGISTRY))
if unknown_models:
    raise ValueError(f'Selected models are not registered: {unknown_models}')

run_framework_self_checks()
suite = unittest.defaultTestLoader.discover(str(PROJECT_DIR / 'tests'))
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
if not test_result.wasSuccessful():
    raise RuntimeError('Pipeline unit tests failed.')
print('Drive project files: PASS')
print('Selected model registry: PASS')
print('Framework self-checks: PASS')
print('Pipeline unit tests: PASS')

## 4. Run or resume

This is safe to rerun. Compatible completed refits load, incomplete work resumes or reruns, pooled files rebuild from refit outputs, and metrics and portfolios refresh.

In [ ]:
# Run one model at a time so pandas never constructs the union of every
# model's lagged feature blocks in memory. Artifacts and resume logic are unchanged.
from dataclasses import replace
import gc
import torch
from src.runner import ExperimentRunner

for model_id in CONFIG.selected_models:
    print(f'\n=== {model_id} ===')
    model_config = replace(CONFIG, selected_models=(model_id,))
    ExperimentRunner(model_config).run()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Restore the full selected-model view for comparison and later cells.
runner = ExperimentRunner(CONFIG)
comparison = runner._cumulative_comparison()
display(comparison)

## 5. Reload saved comparison without training

In [ ]:
import pandas as pd

comparison_path = CONFIG.run_dir / 'model_comparison.csv'
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print('No completed model comparison exists yet.')

## 6. Portfolio implementability robustness

This cached diagnostic reads each pooled prediction file once and never loads model weights. It evaluates the 10% tail portfolio under full/ex-microcap universes, equal/value weighting, fixed proportional transaction-cost scenarios, and the adverse missing-return stress.

In [ ]:
from src.portfolio_robustness import run_portfolio_robustness

portfolio_robustness = run_portfolio_robustness(
    CONFIG.run_dir,
    model_ids=CONFIG.selected_models,
)
display(portfolio_robustness)

## 7. Paired model tests

Run this only after the required models have final monthly diagnostic and portfolio-variant files. The default is strict: a missing planned pair raises an error rather than silently producing an incomplete comparison.

In [ ]:
from src.model_comparison import run_paired_model_comparisons

paired_comparison = run_paired_model_comparisons(CONFIG.run_dir, seed=CONFIG.seed)
display(paired_comparison)

## 8. Final completion checks and frozen outputs

In [ ]:
# Recreate the lightweight runner in case the kernel was restarted or this
# cell is run independently. This does not call runner.run() or train models.
from src.runner import ExperimentRunner
runner = ExperimentRunner(CONFIG)
final_comparison = runner._cumulative_comparison()
expected_models = set(CONFIG.selected_models)
completed_models = set(final_comparison.loc[
    final_comparison['diagnostics_version'].eq(runner.DIAGNOSTICS_VERSION), 'model_id'
])
missing_current = sorted(expected_models - completed_models)
if missing_current:
    raise RuntimeError(f'Models missing current diagnostics: {missing_current}')
expected_oos_months = 12 * (
    CONFIG.universe.end_year - CONFIG.universe.start_year
    - CONFIG.windows.train_years - CONFIG.windows.validation_years + 1
)
selected_comparison = final_comparison.set_index('model_id').loc[list(expected_models)]
if selected_comparison['n_months'].ne(expected_oos_months).any():
    raise RuntimeError('At least one model does not cover the complete OOS portfolio calendar.')
if (selected_comparison['n_signal_months'] + selected_comparison['n_no_signal_months']).ne(expected_oos_months).any():
    raise RuntimeError('At least one model has an incomplete signal-availability accounting.')
required_columns = [
    'robust_oos_r2', 'mean_monthly_rank_ic', 'n_no_signal_months',
    'mean_monthly_calibration_slope', 'tail_5pct_sharpe',
    'tail_10pct_sharpe', 'tail_20pct_sharpe', 'rank_weighted_sharpe',
    'tail_10pct_missing_return_stress_annualized_return',
]
missing_values = final_comparison.set_index('model_id').loc[list(expected_models), required_columns].isna()
if missing_values.any().any():
    print('Legitimate undefined diagnostics require review:')
    display(missing_values[missing_values.any(axis=1)])
else:
    print('All selected models have current complete diagnostics.')
expected_robustness_rows = 4 * len(expected_models)
if len(portfolio_robustness) != expected_robustness_rows:
    raise RuntimeError(
        f'Expected {expected_robustness_rows} robustness rows, got {len(portfolio_robustness)}.'
    )
from src.model_comparison import DEFAULT_MODEL_PAIRS
if len(paired_comparison) != len(DEFAULT_MODEL_PAIRS):
    raise RuntimeError('Paired model comparison is incomplete.')
print('Portfolio robustness and paired model comparisons are complete.')

## Adding a model later

Add a `ModelSpec` and trainer in `src/models.py`, then add its ID to `selected_models`. Model signatures include the relevant source implementation, so a training-code change creates a new signature without overwriting earlier artifacts. Add a focused self-check for every new architecture before starting a full rolling run.